# Chapter 04: Sessions and memory (notebook)

The same code as `main.py`, one step per cell. Keys: with no key in the environment this notebook replays the chapter's recorded fixtures (`tests/fixtures/replay`), no network. With `OPENAI_API_KEY` or the Azure variables set it runs live. The prompts below are the exact strings the fixtures were recorded with; change one and a replay run reports a missing fixture instead of an answer.

In [ ]:
import pathlib
import sys

# Find the repo root (the directory holding tutorials/_shared) and put this chapter on sys.path,
# so the cells below import the chapter's own main.py rather than a copy of it.
HERE = pathlib.Path.cwd().resolve()
ROOT = next(p for p in (HERE, *HERE.parents) if (p / "tutorials" / "_shared").is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(HERE))

import main  # noqa: E402 - the chapter's main.py: its INSTRUCTIONS, tools and helpers are what these cells run

## A session is data

`AgentSession` serialises to a dict and back, which is what lets a conversation survive a process restart.

In [ ]:
import json

from agent_framework import AgentSession

original = AgentSession(session_id="sess-42")
original.state["basket"] = ["SKU-4471"]
restored = AgentSession.from_dict(json.loads(json.dumps(original.to_dict())))
print(restored.session_id, restored.state)

## Turn 1, then throw the agent away

The notebook saves to a temporary file, not the chapter's `session.json`.

In [ ]:
import tempfile

session_file = pathlib.Path(tempfile.mkdtemp()) / "session.json"

agent1 = main.build_agent()
print(await main.ask_and_save(agent1, "Remember: I want to buy SKU-4471.", session_file))
print(session_file.name, "written")

## Turn 2 on a brand-new agent

A second `Agent` object has no memory of its own. It answers from the session loaded from disk.

In [ ]:
agent2 = main.build_agent()
assert agent2 is not agent1
answer = await main.ask_and_save(
    agent2,
    "What did I say I wanted to buy? Answer with the SKU only.",
    session_file,
)
print(answer)
assert "SKU-4471" in answer.upper()